# Notebook 02 — Link Kaggle Healthcare CSV to MPI

**Goal:** Assign each of the 10,000 Kaggle admission rows a `patient_id` from the MPI.

**Linking logic:**
- Match by gender (exact)
- Match by age (±5 years)
- If no match found → fallback to same gender random assignment

**Output:** `data_preparation/linked/admissions_linked.csv`

## 1. Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
import random
import os

KAGGLE_PATH = "../raw/kaggle_healthcare/healthcare_dataset.csv"
MPI_PATH    = "../linked/patients_master.csv"
OUTPUT_DIR  = "../linked/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths OK")

## 2. Load Data

In [ ]:
kaggle = pd.read_csv(KAGGLE_PATH)
mpi    = pd.read_csv(MPI_PATH)

print(f"Kaggle records : {len(kaggle)} rows")
print(f"MPI patients   : {len(mpi)} rows")
print()
print("Kaggle columns:")
print(kaggle.columns.tolist())
print()
print("Kaggle sample:")
kaggle.head(3)

## 3. Normalize Gender Values

In [ ]:
# Kaggle uses 'Male'/'Female', MPI uses 'M'/'F'
# Normalize Kaggle gender to match MPI format
kaggle["gender_normalized"] = kaggle["Gender"].map({"Male": "M", "Female": "F"})

print("Kaggle gender distribution:")
print(kaggle["gender_normalized"].value_counts())
print()
print("MPI gender distribution:")
print(mpi["GENDER"].value_counts())

## 4. Build Gender-Age Lookup from MPI

In [ ]:
# Group MPI patients by gender for fast lookup
mpi_male   = mpi[mpi["GENDER"] == "M"][["patient_id", "age"]].reset_index(drop=True)
mpi_female = mpi[mpi["GENDER"] == "F"][["patient_id", "age"]].reset_index(drop=True)

print(f"MPI male patients   : {len(mpi_male)}")
print(f"MPI female patients : {len(mpi_female)}")

## 5. Link Each Kaggle Row to a Patient ID

In [ ]:
def find_patient_id(gender, age, age_tolerance=5):
    """
    Find a matching patient_id from the MPI.
    Strategy:
      1. Same gender + age within tolerance → pick randomly from matches
      2. Same gender only (fallback) → pick randomly from same gender pool
    Returns (patient_id, match_type)
    """
    pool = mpi_male if gender == "M" else mpi_female

    # Try age match within tolerance
    age_matched = pool[(pool["age"] >= age - age_tolerance) &
                       (pool["age"] <= age + age_tolerance)]

    if len(age_matched) > 0:
        return random.choice(age_matched["patient_id"].tolist()), "gender_age_match"
    else:
        # Fallback: same gender, any age
        return random.choice(pool["patient_id"].tolist()), "gender_only_match"


# Apply to every row
print("Linking Kaggle rows to patient IDs...")
results = kaggle.apply(
    lambda row: find_patient_id(row["gender_normalized"], row["Age"]),
    axis=1
)

kaggle["patient_id"]  = results.apply(lambda x: x[0])
kaggle["match_type"]  = results.apply(lambda x: x[1])

print("Done.")
print()
print("Match type distribution:")
print(kaggle["match_type"].value_counts())

## 6. Clean Up & Final Structure

In [ ]:
# Drop the temporary normalized gender column
kaggle.drop(columns=["gender_normalized"], inplace=True)

# Reorder columns — patient_id and match_type first
cols = ["patient_id", "match_type"] + [c for c in kaggle.columns if c not in ["patient_id", "match_type"]]
kaggle = kaggle[cols]

print("Final columns:")
print(kaggle.columns.tolist())
print()
kaggle.head(5)

## 7. Quality Check

In [ ]:
print("=== Admissions Linking Quality Check ===")
print(f"Total admission records     : {len(kaggle)}")
print(f"Unique patients assigned    : {kaggle['patient_id'].nunique()}")
print(f"Patients with 0 admissions  : {len(mpi) - kaggle['patient_id'].nunique()}")
print()
print("Match type distribution:")
print(kaggle["match_type"].value_counts())
print()
print("Admissions per patient (stats):")
admissions_per_patient = kaggle.groupby("patient_id").size()
print(admissions_per_patient.describe())
print()
print("Top 5 most admitted patients:")
print(admissions_per_patient.sort_values(ascending=False).head())
print()
print("Medical condition distribution:")
print(kaggle["Medical Condition"].value_counts())

## 8. Save Output

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "admissions_linked.csv")
kaggle.to_csv(output_path, index=False)

print(f"Saved → {output_path}")
print(f"Shape  : {kaggle.shape}")